# EP1 Machine Learning — Spotify Tracks
## Inteligencia musical y predicción de popularidad

**Asignatura:** MLY1101 Machine Learning | **Institución:** Duoc UC | **Metodología:** CRISP-DM

**Objetivo:** preparar los datos y realizar el análisis exploratorio para predecir
`popularidad` (regresión, escala 0–100). El modelo final corresponde a EP2.

**Actualización:** nombres de columnas en español desde la carga, tratamiento de
nulos, «?» y ceros según las reglas del proyecto, auditoría de filas afectadas y
moda del compás aprendida exclusivamente con entrenamiento.

**Estado de esta entrega:** el CSV original no fue adjuntado. Las celdas se entregan
sin resultados guardados; los conteos, estadísticas y gráficos se recalculan al
ejecutar todo el notebook con `Spotify_Tracks_Dataset.csv`. No se presentan datos
sintéticos de validación como resultados del proyecto.


## 0. Importaciones, nombres en español y configuración


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)
# display() está disponible en Jupyter y Google Colab.
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

# Nombres de datos en español desde la primera lectura del CSV.
# Sin tildes ni espacios en el código; etiquetas legibles para la presentación.
COLUMNAS_ESPANOL = {
    'track_id': 'id_cancion', 'artists': 'artistas',
    'album_name': 'nombre_album', 'track_name': 'nombre_cancion',
    'popularity': 'popularidad', 'duration_ms': 'duracion_ms',
    'explicit': 'contenido_explicito', 'danceability': 'bailabilidad',
    'energy': 'energia', 'key': 'tonalidad', 'loudness': 'volumen_db',
    'mode': 'modo', 'speechiness': 'presencia_habla',
    'acousticness': 'acusticidad', 'instrumentalness': 'instrumentalidad',
    'liveness': 'presencia_en_vivo', 'valence': 'positividad',
    'tempo': 'tempo_bpm', 'time_signature': 'compas',
    'track_genre': 'genero_musical',
}
ETIQUETAS = {
    'id_cancion': 'ID de canción', 'artistas': 'Artistas',
    'nombre_album': 'Nombre del álbum', 'nombre_cancion': 'Nombre de la canción',
    'popularidad': 'Popularidad', 'duracion_ms': 'Duración (ms)',
    'duracion_min': 'Duración (min)', 'contenido_explicito': 'Contenido explícito',
    'bailabilidad': 'Bailabilidad', 'energia': 'Energía',
    'tonalidad': 'Tonalidad', 'volumen_db': 'Volumen (dB)', 'modo': 'Modo musical',
    'presencia_habla': 'Presencia de habla', 'acusticidad': 'Acusticidad',
    'instrumentalidad': 'Instrumentalidad', 'presencia_en_vivo': 'Presencia en vivo',
    'positividad': 'Positividad musical', 'tempo_bpm': 'Tempo (BPM)',
    'compas': 'Compás', 'genero_musical': 'Género musical',
}

# Opcional: indica aquí la ubicación exacta del CSV (por ejemplo, en Colab).
RUTA_DATOS = None  # Ejemplo: Path('/content/Spotify_Tracks_Dataset.csv')
carpeta_actual = Path.cwd()
PROJECT_ROOT = carpeta_actual.parent if carpeta_actual.name == 'notebooks' else carpeta_actual
candidatas = [
    PROJECT_ROOT / 'data' / 'raw' / 'Spotify_Tracks_Dataset.csv',
    carpeta_actual / 'Spotify_Tracks_Dataset.csv',
    carpeta_actual / 'upload' / 'Spotify_Tracks_Dataset.csv',
]
DATA_PATH = (Path(RUTA_DATOS).expanduser() if RUTA_DATOS is not None
             else next((p for p in candidatas if p.is_file()), candidatas[0]))
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
IMAGES_DIR = PROJECT_ROOT / 'images'
print('Proyecto:', PROJECT_ROOT)
print('Dataset:', DATA_PATH)


## 1. Metodología CRISP-DM

El proyecto sigue CRISP-DM. Esta EP1 cubre las **fases 1–3**:

| Fase | Descripción | Estado EP1 |
|------|-------------|------------|
| 1. Business Understanding | Problema, objetivos, KPIs | ✅ Completo |
| 2. Data Understanding | EDA, calidad, relaciones | ✅ Completo |
| 3. Data Preparation | Limpieza, transformaciones, pipeline | ✅ Completo |
| 4. Modeling | Entrenamiento de modelos | 🔄 EP2 |
| 5. Evaluation | Métricas y comparación | 🔄 EP2 |
| 6. Deployment | Scoring en producción | 🔄 EP3 |


## 2. Problema de negocio, objetivos y KPIs

### Problema de negocio
Un equipo de inteligencia musical necesita determinar si los **atributos medibles de audio** de una canción permiten anticipar su nivel de popularidad para apoyar decisiones de curaduría editorial y promoción algorítmica.

### Objetivo general
Construir una base analítica reproducible para desarrollar un modelo de regresión que estime `popularidad` a partir de features musicales.

### Objetivos específicos
- **Analítico:** identificar qué atributos se asocian con mayor popularidad.
- **ML:** entrenar un modelo de regresión con `popularidad` como target.

### KPIs
| KPI | Umbral de referencia |
|-----|---------------------|
| Cobertura de datos (sin nulos en features) | ≥ 99% |
| Contaminación train/test (id_cancion compartidos) | 0% |
| MAE (fase futura) | < 10 puntos |
| RMSE (fase futura) | < 15 puntos |
| R² (fase futura) | > 0.20 |


## 3. Fuentes de datos y herramientas colaborativas

**Fuente:** `Spotify_Tracks_Dataset.csv` — Kaggle (*Spotify Tracks Dataset*, MaharshiPandya).

**Limitación:** sin fecha de extracción ni versión de API. La `popularidad` puede estar desactualizada.

| Herramienta | Uso |
|-------------|-----|
| Python / pandas / numpy | Manipulación de datos |
| matplotlib / seaborn | Visualización |
| scikit-learn | Pipeline ML |
| Jupyter Notebook | Documentación reproducible |
| Git / GitHub | Control de versiones grupal |
| Google Colab | Ejecución compartida |


## 4. Carga y estructura del dataset

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f'No se encontró {DATA_PATH}. Agrega Spotify_Tracks_Dataset.csv en '
        'data/raw/ o junto al notebook, o configura RUTA_DATOS en la sección 0.'
    )

# Renombrar ANTES de cualquier head(), tabla o gráfico.
df_crudo = pd.read_csv(DATA_PATH).rename(columns=COLUMNAS_ESPANOL)
df_crudo = df_crudo.drop(columns=['Unnamed: 0'], errors='ignore')
faltan_columnas = sorted(set(COLUMNAS_ESPANOL.values()) - set(df_crudo.columns))
if faltan_columnas:
    raise ValueError(f'Faltan columnas requeridas en el CSV: {faltan_columnas}')
if df_crudo.columns.duplicated().any():
    raise ValueError('Hay nombres de columna duplicados después de traducir.')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
print(f'Datos cargados: {len(df_crudo):,} filas × {df_crudo.shape[1]} columnas útiles')
display(df_crudo.head(3))


## 5. Diccionario y clasificación analítica de variables

| Variable | Tipo | Rol | Descripción |
|----------|------|-----|-------------|
| `id_cancion` | Texto | Identificador | ID único de Spotify — excluir del modelo |
| `artistas` | Texto | Metadato | Nombre(s) del artista |
| `nombre_album` | Texto | Metadato | Nombre del álbum |
| `nombre_cancion` | Texto | Metadato | Nombre de la canción |
| `popularidad` | Numérica (int) | **TARGET** | Popularidad 0–100 |
| `duracion_ms` | Numérica | Feature | Duración en ms |
| `contenido_explicito` | Booleana | Feature | Contenido explícito |
| `bailabilidad` | Float 0–1 | Feature | Aptitud para bailar |
| `energia` | Float 0–1 | Feature | Intensidad percibida |
| `tonalidad` | Categórica | Feature | Tonalidad (0=C…11=B) |
| `volumen_db` | Float (dB) | Feature | Volumen promedio |
| `modo` | Categórica | Feature | Mayor(1)/Menor(0) |
| `presencia_habla` | Float 0–1 | Feature | Presencia de voz hablada |
| `acusticidad` | Float 0–1 | Feature | Nivel acústico |
| `instrumentalidad` | Float 0–1 | Feature | Nivel instrumental |
| `presencia_en_vivo` | Float 0–1 | Feature | Presencia de audiencia en vivo |
| `positividad` | Float 0–1 | Feature | Positividad musical |
| `tempo_bpm` | Float (BPM) | Feature | Velocidad |
| `compas` | Categórica | Feature | Compás musical |
| `genero_musical` | Categórica | Feature | Género (114 categorías) |

> **Nota:** `tonalidad`, `modo` y `compas` son **códigos musicales**, no magnitudes. Se tratarán como categóricas.

### Equivalencias con el archivo original

| Original | Español |
|---|---|
| `track_id` | `id_cancion` |
| `artists` | `artistas` |
| `album_name` | `nombre_album` |
| `track_name` | `nombre_cancion` |
| `popularity` | `popularidad` |
| `duration_ms` | `duracion_ms` |
| `explicit` | `contenido_explicito` |
| `danceability` | `bailabilidad` |
| `energy` | `energia` |
| `key` | `tonalidad` |
| `loudness` | `volumen_db` |
| `mode` | `modo` |
| `speechiness` | `presencia_habla` |
| `acousticness` | `acusticidad` |
| `instrumentalness` | `instrumentalidad` |
| `liveness` | `presencia_en_vivo` |
| `valence` | `positividad` |
| `tempo` | `tempo_bpm` |
| `time_signature` | `compas` |
| `track_genre` | `genero_musical` |


In [ ]:
# Inspección de estructura
estructura = pd.DataFrame({
    'variable': df_crudo.columns,
    'tipo_dato':    df_crudo.dtypes.astype(str).values,
    'no_nulos': df_crudo.notna().sum().values,
    'nulos':    df_crudo.isna().sum().values,
    'unicos':   df_crudo.nunique(dropna=True).values
})
display(estructura)


## 6. Limpieza y transformaciones justificadas

| Variables en español | Condición | Tratamiento |
|---|---|---|
| `artistas`, `nombre_cancion` | Nulo, vacío o «?» | Eliminar la fila |
| `nombre_album` | Nulo, vacío o «?» | Imputar `DESCONOCIDO` en las filas conservadas |
| `tempo_bpm`, `bailabilidad`, `energia` | Al menos una igual a cero | Eliminar la fila una sola vez |
| Las tres métricas anteriores | Nulo explícito | Eliminar, sin KNN en esta versión |
| `compas` | Cero o nulo | Moda de los valores observados **solo en entrenamiento** |
| `duracion_ms` | Cero, negativo o nulo | Eliminar la fila |

- «?» y espacios vacíos se normalizan solo en los tres metadatos de texto.
- No se asigna `SINGLE`: la falta de nombre no demuestra que un lanzamiento sea un single.
- Los conteos informados (1 nulo de texto, 20 «?», 157 ceros de audio, 163 compases
  cero y 1 duración cero) son referencias a comprobar. Las filas pueden coincidir;
  **no se suman los conteos para calcular las eliminaciones**.
- Eliminar ceros de bailabilidad y energía es una **decisión de este proyecto**:
  cero pertenece a sus escalas y no demuestra un dato faltante. Puede sesgar la
  muestra hacia canciones más bailables y enérgicas.
- Se mantienen los ceros válidos de otras variables, como `popularidad`, `modo`,
  `tonalidad` e `instrumentalidad`.
- No se usa `KNNImputer` para compás: su promedio podría crear categorías
  fraccionarias. La moda conserva un valor observado; en empate se elige el menor.
- Los compases observados positivos se conservan si son enteros. Las categorías
  inusuales se reportan para revisión, sin inventar valores ni truncar decimales.
- Los valores extremos detectados por IQR se reportan; no se eliminan por IQR.

La partición por canción se reserva aquí para aprender la moda sin consultar el
conjunto de prueba. `df_base` conserva el compás sin imputar para el pipeline;
`df_limpio` incorpora esa moda para EDA y exportación. En EP2 se debe volver a
ajustar el pipeline dentro de cada pliegue de validación usando `df_base`.

Referencias: [escalas de audio de Spotify](https://developer.spotify.com/documentation/web-api/reference/get-audio-features),
[moda con SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html).


In [ ]:
def limpiar_spotify(datos, semilla=42):
    """Limpieza auditable, sin modificar la entrada ni aprender del test."""
    df = datos.copy()
    textos = ['artistas', 'nombre_album', 'nombre_cancion']
    audio_cero = ['tempo_bpm', 'bailabilidad', 'energia']

    # Diagnóstico ANTES de eliminar filas: separa nulos, «?» y ceros.
    auditoria = pd.DataFrame(index=textos + audio_cero + ['compas', 'duracion_ms'])
    auditoria.index.name = 'variable'
    auditoria['nulos_explicitos'] = df[auditoria.index].isna().sum()
    auditoria['interrogaciones'] = 0
    auditoria['vacios'] = 0
    auditoria['ceros'] = 0
    for col in textos:
        texto = df[col].astype('string').str.strip()
        auditoria.loc[col, 'interrogaciones'] = int(texto.eq('?').sum())
        auditoria.loc[col, 'vacios'] = int(texto.eq('').sum())
        df[col] = texto.mask(texto.isin(['', '?']), pd.NA)
    for col in audio_cero + ['compas', 'duracion_ms']:
        df[col] = pd.to_numeric(df[col], errors='raise').astype(float)
        auditoria.loc[col, 'ceros'] = int(df[col].eq(0).sum())

    motivos = pd.DataFrame({
        'artista_o_cancion_faltante': df[['artistas', 'nombre_cancion']].isna().any(axis=1),
        'cero_en_audio': df[audio_cero].eq(0).any(axis=1),
        'nulo_en_audio': df[audio_cero].isna().any(axis=1),
        'duracion_invalida': df['duracion_ms'].isna() | df['duracion_ms'].le(0),
    }, index=df.index)
    eliminar = motivos.any(axis=1)
    resumen = pd.DataFrame({
        'criterio': motivos.columns,
        'filas_afectadas_pueden_coincidir': motivos.sum().values,
    })
    # Una fila con varias incidencias se elimina solo una vez.
    base = df.loc[~eliminar].copy()
    if base.empty:
        raise ValueError('No quedan filas después de la limpieza; revisa las incidencias.')
    albumes_imputados = int(base['nombre_album'].isna().sum())
    base['nombre_album'] = base['nombre_album'].fillna('DESCONOCIDO')
    base['compas'] = base['compas'].mask(base['compas'].eq(0), np.nan)
    observados = base['compas'].dropna()
    if ((observados <= 0) | ~np.isfinite(observados) | (observados % 1 != 0)).any():
        raise ValueError('Hay compases negativos, no finitos o decimales. Revisarlos sin truncar.')
    base['duracion_min'] = base['duracion_ms'] / 60000

    ids = base['id_cancion'].astype('string').str.strip()
    if ids.isna().any() or ids.isin(['', '?']).any():
        raise ValueError('Hay identificadores de canción faltantes; no se puede agrupar con seguridad.')
    base['id_cancion'] = ids
    if ids.nunique() < 2:
        raise ValueError('Se necesitan al menos dos canciones distintas para separar entrenamiento y prueba.')
    divisor = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=semilla)
    idx_entrenamiento, idx_prueba = next(divisor.split(base, groups=ids))
    compas_entrenamiento = base.iloc[idx_entrenamiento][['compas']]
    if compas_entrenamiento['compas'].notna().sum() == 0:
        raise ValueError('No hay compases observados en entrenamiento para calcular la moda.')
    imputador = SimpleImputer(strategy='most_frequent')
    imputador.fit(compas_entrenamiento)
    moda = int(imputador.statistics_[0])
    limpio = base.copy()
    limpio['compas'] = imputador.transform(base[['compas']]).ravel().astype('int64')

    particion = pd.Series('prueba', index=base.index, name='particion')
    particion.iloc[idx_entrenamiento] = 'entrenamiento'
    imputaciones = pd.DataFrame({
        'particion': particion,
        'compas_imputado': base['compas'].isna(),
    }).groupby('particion')['compas_imputado'].sum().astype(int)
    balance = pd.DataFrame({
        'indicador': ['filas_originales', 'filas_eliminadas_sin_duplicar', 'filas_conservadas',
                      'albumes_imputados_en_filas_conservadas', 'compases_imputados',
                      'moda_compas_entrenamiento'],
        'cantidad': [len(df), int(eliminar.sum()), len(limpio), albumes_imputados,
                     int(base['compas'].isna().sum()), moda],
    })
    descartadas = datos.loc[eliminar].join(motivos.loc[eliminar].add_prefix('motivo_'))
    return base, limpio, idx_entrenamiento, idx_prueba, auditoria, resumen, balance, imputaciones, descartadas

(df_base, df_limpio, train_idx, test_idx, auditoria_faltantes, resumen_eliminacion,
 balance_limpieza, imputaciones_compas, filas_descartadas) = limpiar_spotify(df_crudo)

display(auditoria_faltantes)
display(resumen_eliminacion)
display(balance_limpieza)
display(imputaciones_compas.to_frame('cantidad_imputada'))
perdida_pct = 100 * (len(df_crudo) - len(df_limpio)) / len(df_crudo)
print(f'Pérdida real de filas: {perdida_pct:.4f}% (sin duplicar coincidencias)')

# Guardar datos limpios, base sin imputación aprendida y trazabilidad.
df_limpio.to_csv(PROCESSED_DIR / 'spotify_clean.csv', index=False)
df_base.to_csv(PROCESSED_DIR / 'spotify_base_modelo.csv', index=False)
auditoria_faltantes.to_csv(PROCESSED_DIR / 'auditoria_faltantes.csv')
resumen_eliminacion.to_csv(PROCESSED_DIR / 'resumen_eliminacion.csv', index=False)
balance_limpieza.to_csv(PROCESSED_DIR / 'balance_limpieza.csv', index=False)
filas_descartadas.to_csv(PROCESSED_DIR / 'filas_descartadas.csv', index_label='indice_origen')
particion = pd.DataFrame({'id_cancion': df_base['id_cancion'], 'particion': 'prueba'})
particion.iloc[train_idx, particion.columns.get_loc('particion')] = 'entrenamiento'
particion.to_csv(PROCESSED_DIR / 'particion_modelo.csv', index_label='indice_origen')
print('Datos y auditoría guardados; todos los nombres de columna están en español.')


In [ ]:
# Clasificación de variables para el análisis
audio_numericas = [
    'duracion_ms', 'bailabilidad', 'energia', 'volumen_db', 'presencia_habla',
    'acusticidad', 'instrumentalidad', 'presencia_en_vivo', 'positividad', 'tempo_bpm'
]
variables_categoricas = ['contenido_explicito', 'tonalidad', 'modo', 'compas', 'genero_musical']
metadatos_texto        = ['id_cancion', 'artistas', 'nombre_album', 'nombre_cancion']
numericas_descriptivas = ['popularidad', 'duracion_ms', 'duracion_min'] + audio_numericas[1:]

print('Numéricas predictoras:', audio_numericas)
print('Categóricas predictoras:', variables_categoricas)
print('Metadatos (excluir del modelo):', metadatos_texto)


## 7. Estadística descriptiva completa

Se calculan media, mediana, desviación estándar, percentiles 25/50/75/90/95/99, rango, skewness y curtosis para cada variable numérica.


In [ ]:
# Estadística descriptiva extendida con skewness y curtosis
rows = []
for col in numericas_descriptivas:
    s = df_limpio[col].dropna()
    q25, q50, q75, q90, q95, q99 = s.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    rows.append({
        'variable':   col,
        'cantidad':      int(len(s)),
        'media':       s.mean(),
        'mediana':     s.median(),
        'desviacion_estandar':        s.std(),
        'minimo':        s.min(),
        'P25':        q25,
        'P75':        q75,
        'P90':        q90,
        'P95':        q95,
        'P99':        q99,
        'maximo':        s.max(),
        'rango':      s.max() - s.min(),
        'asimetria':   s.skew(),
        'curtosis':   s.kurtosis()
    })

stats_df = pd.DataFrame(rows).set_index('variable')
display(stats_df.round(4))
stats_df.to_csv(PROCESSED_DIR / 'descriptive_statistics.csv')
print('Estadísticas guardadas.')


### Interpretación de la tabla actualizada

Las cifras deben salir de la ejecución actual, porque la eliminación de ceros
cambia medias, percentiles y correlaciones. Comparar media y mediana permite
describir la distribución; asimetría y curtosis ayudan a revisar valores extremos.

No confundir una correlación débil con la imposibilidad de predecir: pueden existir
relaciones no lineales o interacciones. El rendimiento se comprobará en EP2.


## 8. Histogramas y distribuciones

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
axes = axes.flatten()

for i, col in enumerate(numericas_descriptivas):
    ax = axes[i]
    data = df_limpio[col].dropna()
    ax.hist(data, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(ETIQUETAS[col], fontsize=11)
    ax.set_xlabel(ETIQUETAS[col], fontsize=9)
    ax.set_ylabel('Frecuencia', fontsize=9)
    # Añadir líneas de media y mediana
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Media: {data.mean():.2f}')
    ax.axvline(data.median(), color='orange', linestyle='-',  linewidth=1.5, label=f'Mediana: {data.median():.2f}')
    ax.legend(fontsize=7)

# Ocultar subplots vacíos
for j in range(len(numericas_descriptivas), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuciones de variables numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_histogramas_numericas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado: 01_histogramas_numericas.png')


## 9. Variables categóricas: frecuencias y proporciones

Se analiza la distribución de `contenido_explicito`, `tonalidad`, `modo`, `compas` y `genero_musical`.


In [ ]:
def tabla_frecuencias(data, variable):
    counts = data[variable].value_counts(dropna=False)
    pct    = data[variable].value_counts(dropna=False, normalize=True) * 100
    return pd.DataFrame({'frecuencia': counts, 'proporcion_pct': pct.round(2)})

for variable in ['contenido_explicito', 'tonalidad', 'modo', 'compas']:
    print(f'\n── {variable} ──')
    display(tabla_frecuencias(df_limpio, variable))

print(f'\n── genero_musical: {df_limpio["genero_musical"].nunique()} géneros únicos (mostrando top 20) ──')
display(tabla_frecuencias(df_limpio, 'genero_musical').head(20))


In [ ]:
# Gráficos de variables categóricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, variable in zip(axes.flatten(), ['contenido_explicito', 'tonalidad', 'modo', 'compas']):
    counts = df_limpio[variable].value_counts().sort_index()
    ax.bar([({False: 'No', True: 'Sí'}.get(v, str(v)) if variable == 'contenido_explicito'
            else {0: 'Menor', 1: 'Mayor'}.get(v, str(v)) if variable == 'modo'
            else str(v)) for v in counts.index], counts.values, color='steelblue', edgecolor='black')
    ax.set_title(f'Frecuencia: {ETIQUETAS[variable]}', fontsize=12, fontweight='bold')
    ax.set_xlabel(ETIQUETAS[variable], fontsize=10)
    ax.set_ylabel('Frecuencia', fontsize=10)
    ax.set_ylim(0, max(1, counts.max()) * 1.15)
    for bar, val in zip(ax.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max() * 0.015,
                f'{val:,}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribución de variables categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '02_variables_categoricas.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Calidad de datos: dominios y duplicados

Se comprueba el resultado de la limpieza y se reportan valores inusuales. Las
reglas específicas del proyecto se distinguen de los dominios de las variables.
Los compases 1 y 2, si aparecen en el CSV histórico, se conservan como categorías
observadas pero se señalan: la documentación actual de Spotify describe 3–7.


In [ ]:
checks = {
    'Nulos después de limpiar (todas las columnas)': int(df_limpio.isna().sum().sum()),
    'Artista o canción vacíos / interrogación': int(df_limpio[['artistas', 'nombre_cancion']].isin(['', '?']).sum().sum()),
    'Álbum vacío / interrogación': int(df_limpio['nombre_album'].isin(['', '?']).sum()),
    'Popularidad fuera de 0–100': int((~df_limpio['popularidad'].between(0, 100)).sum()),
    'Tonalidad fuera de códigos -1 a 11': int((~df_limpio['tonalidad'].isin(range(-1, 12))).sum()),
    'Modo fuera de {0, 1}': int((~df_limpio['modo'].isin([0, 1])).sum()),
    'Duración no positiva': int(df_limpio['duracion_ms'].le(0).sum()),
    'Tempo no positivo': int(df_limpio['tempo_bpm'].le(0).sum()),
    'Bailabilidad o energía cero (regla del proyecto)': int(df_limpio[['bailabilidad', 'energia']].eq(0).any(axis=1).sum()),
    'Compás nulo, no positivo o no entero': int((df_limpio['compas'].isna() | df_limpio['compas'].le(0) | df_limpio['compas'].mod(1).ne(0)).sum()),
    'Valores numéricos no finitos': int((~np.isfinite(df_limpio[audio_numericas + ['popularidad']])).sum().sum()),
}
for col in ['bailabilidad', 'energia', 'presencia_habla', 'acusticidad',
            'instrumentalidad', 'presencia_en_vivo', 'positividad']:
    checks[f'{ETIQUETAS[col]} fuera de 0–1'] = int((~df_limpio[col].between(0, 1)).sum())
checks_df = pd.DataFrame(list(checks.items()), columns=['Regla', 'Cantidad'])
checks_df['Estado'] = np.where(checks_df['Cantidad'].eq(0), 'OK', 'Revisar')
display(checks_df)
if checks_df['Cantidad'].gt(0).any():
    raise ValueError('Persisten incidencias de calidad; revisarlas antes del modelamiento.')

advertencias = pd.DataFrame({
    'Observación': ['Volumen positivo en dB: revisar fuente', 'Compás fuera del rango 3–7 documentado actualmente'],
    'Cantidad': [int(df_limpio['volumen_db'].gt(0).sum()), int((~df_limpio['compas'].between(3, 7)).sum())],
})
display(advertencias)
print(f'Duplicados exactos: {df_limpio.duplicated().sum():,}')
conteos_id = df_limpio['id_cancion'].value_counts()
print(f'Canciones únicas: {len(conteos_id):,}')
print(f'IDs con más de una fila: {conteos_id.gt(1).sum():,}')
print(f'Filas adicionales respecto de IDs únicos: {len(df_limpio) - len(conteos_id):,}')
print('La separación por ID evita compartir canciones entre entrenamiento y prueba; no elimina duplicados.')


## 11. Outliers mediante IQR

El criterio IQR se usa para **detectar**, no para eliminar automáticamente.
Cada caso se evalúa individualmente antes de tomar una decisión.


In [ ]:
outlier_rows = []
for col in audio_numericas + ['popularidad']:
    s = df_limpio[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low  = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    mask = (s < low) | (s > high)
    outlier_rows.append({
        'variable':      col,
        'Q1':            q1,
        'Q3':            q3,
        'limite_inf':    low,
        'limite_sup':    high,
        'n_outliers':    int(mask.sum()),
        'pct_outliers':  round(mask.mean() * 100, 2),
        'decision':      'Conservar'
    })

outliers_df = pd.DataFrame(outlier_rows).sort_values('pct_outliers', ascending=False)
display(outliers_df.round(3))
outliers_df.to_csv(PROCESSED_DIR / 'outliers_iqr_summary.csv', index=False)
print('\nResumen de outliers guardado.')


### Justificación de las decisiones

El IQR identifica valores alejados del centro de la distribución, pero no demuestra
que sean errores. Se conservan los valores extremos para revisar su contexto
musical. Duraciones largas, grabaciones instrumentales o con habla pueden ser
reales; no se afirma que todos los extremos hayan sido verificados individualmente.

Las eliminaciones de ceros de audio y duración corresponden a las reglas de la
sección 6, aplicadas **antes** del IQR. Sus cantidades no se mezclan con outliers.


## 12. Correlaciones y relaciones entre variables

In [ ]:
corr_cols = audio_numericas + ['popularidad']
corr = df_limpio[corr_cols].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr.rename(index=ETIQUETAS, columns=ETIQUETAS), mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8})
ax.set_title('Matriz de correlación de variables numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '03_matriz_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlaciones con popularidad
corr_pop = corr['popularidad'].drop('popularidad').sort_values(key=lambda s: s.abs(), ascending=False)
print('=== Correlaciones con popularidad (ordenadas por valor absoluto) ===')
display(corr_pop.to_frame('correlacion_con_popularidad').round(4))
print(f'Correlaciones con |r| >= 0.10: {corr_pop.abs().ge(0.10).sum()} de {corr_pop.notna().sum()} calculables.')
print('La correlación describe asociación lineal, no causalidad ni desempeño del modelo.')


## 13. Comparaciones categóricas: popularidad por grupos

In [ ]:
# Por género
estadisticas_genero = (
    df_limpio.groupby('genero_musical')
    .agg(
        n               = ('id_cancion', 'size'),
        popularidad_media = ('popularidad', 'mean'),
        popularidad_mediana  = ('popularidad', 'median'),
        popularidad_desviacion  = ('popularidad', 'std'),
        bailabilidad_media      = ('bailabilidad', 'mean'),
        energia_media     = ('energia', 'mean')
    )
    .sort_values('popularidad_media', ascending=False)
    .round(2)
)

print('=== 10 primeros géneros por popularidad media ===')
display(estadisticas_genero.head(10))
print('\n=== 10 últimos géneros por popularidad media ===')
display(estadisticas_genero.tail(10))

estadisticas_genero.to_csv(PROCESSED_DIR / 'genre_popularity_stats.csv')


In [ ]:
# Gráfico Top 15 y Bottom 15
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

top15  = estadisticas_genero.head(15).sort_values('popularidad_media')
bot15  = estadisticas_genero.tail(15).sort_values('popularidad_media', ascending=False)

ax1.barh(top15.index, top15['popularidad_media'], color='steelblue', edgecolor='black')
ax1.set_title('15 géneros con mayor media\n(mayor popularidad media)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Popularidad media')
ax1.axvline(df_limpio['popularidad'].mean(), color='red', linestyle='--', label=f'Media global ({df_limpio["popularidad"].mean():.1f})')
ax1.legend(fontsize=9)

ax2.barh(bot15.index, bot15['popularidad_media'], color='salmon', edgecolor='black')
ax2.set_title('15 géneros con menor media\n(menor popularidad media)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Popularidad media')
ax2.axvline(df_limpio['popularidad'].mean(), color='red', linestyle='--', label=f'Media global')
ax2.legend(fontsize=9)

plt.suptitle('Popularidad media por género', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '04_popularidad_por_genero.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Por contenido_explicito
estadisticas_explicito = df_limpio.groupby('contenido_explicito')['popularidad'].agg(['count','mean','median','std']).round(2)
print('=== Popularidad según contenido explícito ===')
display(estadisticas_explicito.rename(columns={'count': 'cantidad', 'mean': 'media', 'median': 'mediana', 'std': 'desviacion_estandar'}))

fig, ax = plt.subplots(figsize=(7, 5))
groups_data = [df_limpio.loc[df_limpio['contenido_explicito'] == False, 'popularidad'].values,
               df_limpio.loc[df_limpio['contenido_explicito'] == True,  'popularidad'].values]
bp = ax.boxplot(groups_data, positions=[1, 2], patch_artist=True,
                showfliers=False)
ax.set_xticks([1, 2], ['No explícita', 'Explícita'])
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('salmon')
ax.set_title('Popularidad según contenido explícito', fontsize=12, fontweight='bold')
ax.set_ylabel('Popularidad')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '05_popularidad_explicit.png', dpi=150, bbox_inches='tight')
plt.show()
if {False, True}.issubset(estadisticas_explicito.index):
    print('Diferencia de medias: {:.2f} puntos (no implica causalidad)'.format(
        estadisticas_explicito.loc[True, 'mean'] - estadisticas_explicito.loc[False, 'mean']))


## 14. Análisis adicional: perfil de canciones populares

Se comparan los features de audio entre canciones con `popularidad ≥ 70` y el resto.


In [ ]:
pop_alta  = df_limpio[df_limpio['popularidad'] >= 70]
pop_baja  = df_limpio[df_limpio['popularidad'] < 70]

print(f'Canciones con popularidad >= 70: {len(pop_alta):,} ({len(pop_alta)/len(df_limpio)*100:.1f}%)')
print(f'Resto: {len(pop_baja):,}')

comparacion = pd.DataFrame({
    'Popular (≥70)':   pop_alta[audio_numericas].mean(),
    'Resto (<70)':     pop_baja[audio_numericas].mean(),
}).round(4)
comparacion['Diferencia'] = (comparacion['Popular (≥70)'] - comparacion['Resto (<70)']).round(4)
display(comparacion)
print('Comparar las diferencias calculadas arriba; no implican causalidad.')


## 15. Preparación para Machine Learning

- **Objetivo:** `popularidad`; excluir identificadores, metadatos de alta cardinalidad
  y `duracion_min` por duplicar `duracion_ms`.
- **Numéricas:** `StandardScaler`, sensible a valores extremos. No es un escalador robusto.
- **Compás:** `SimpleImputer(strategy='most_frequent')` y `OneHotEncoder`.
- **Otras categóricas:** `OneHotEncoder(handle_unknown='ignore')`.
- **Partición:** reutilizar los índices por `id_cancion` reservados en la sección 6.
  La fracción de prueba es 20% de los grupos, no necesariamente 20% exacto de filas.
- **Sin fuga de información:** el pipeline recibe `df_base`, con compás sin imputar,
  y aprende exclusivamente de entrenamiento. El conjunto de prueba no decide la moda.

En EP2, mantener la imputación dentro del pipeline en cada pliegue de validación
por grupos. No usar `spotify_clean.csv` ya imputado para ajustar de nuevo una
validación cruzada. El EDA completo es descriptivo: no usar hallazgos del conjunto
de prueba para seleccionar modelos ni variables.


In [ ]:
model_numeric = audio_numericas.copy()
model_categorical = ['contenido_explicito', 'tonalidad', 'modo', 'compas', 'genero_musical']
otras_categoricas = [c for c in model_categorical if c != 'compas']

# Usar la base que todavía conserva NaN en compás, no la exportación imputada.
X = df_base[model_numeric + model_categorical].copy()
y = df_base['popularidad'].copy()
groups = df_base['id_cancion'].copy()
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
train_ids, test_ids = set(groups.iloc[train_idx]), set(groups.iloc[test_idx])
overlap = train_ids & test_ids
assert not overlap, 'Existe fuga de información por ID de canción.'
assert len(X_train) + len(X_test) == len(df_base)
print(f'Entrenamiento: {X_train.shape}; prueba: {X_test.shape}')
print(f'ID de canción compartidos: {len(overlap)}')


In [ ]:
transformador_compas = Pipeline([
    ('imputador', SimpleImputer(strategy='most_frequent')),
    ('codificador', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ('numericas', StandardScaler(), model_numeric),
    ('categoricas', OneHotEncoder(handle_unknown='ignore', sparse_output=False), otras_categoricas),
    ('compas', transformador_compas, ['compas']),
])

# Ajuste SOLO en entrenamiento. En esta versión no se entrena un predictor.
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)
moda_pipeline = int(preprocessor.named_transformers_['compas'].named_steps['imputador'].statistics_[0])
moda_limpieza = int(balance_limpieza.loc[
    balance_limpieza['indicador'].eq('moda_compas_entrenamiento'), 'cantidad'
].iloc[0])
assert moda_pipeline == moda_limpieza, 'La imputación debe coincidir con la moda reservada.'
assert np.isfinite(X_train_prep).all() and np.isfinite(X_test_prep).all()
assert X_train_prep.shape[1] == X_test_prep.shape[1]
nombres_preparados = preprocessor.get_feature_names_out()
pd.DataFrame({'variable_transformada': nombres_preparados}).to_csv(
    PROCESSED_DIR / 'variables_modelo.csv', index=False
)
print(f'Moda del compás aprendida solo con entrenamiento: {moda_pipeline}')
print(f'Matriz de entrenamiento: {X_train_prep.shape}')
print(f'Matriz de prueba: {X_test_prep.shape}')
print('Verificado: matrices sin nulos ni infinitos y sin canciones compartidas entre conjuntos.')


## 16. Sesgos, ética y privacidad

### Sesgos identificados

| Sesgo | Tipo | Descripción |
|-------|------|-------------|
| **Muestreo artificial balanceado** | Observado | El documento previo reporta 1.000 canciones por género; la limpieza puede alterar ese balance |
| **Feedback loop** | Potencial | Popularidad predicha → más promoción → más reproducciones → ciclo autoreforzante |
| **Sesgo de exposición por género** | Potencial | Géneros no anglosajones tienen menor popularidad, no necesariamente por calidad |
| **Sesgo por artista** | Observado | Artistas con más discografía tienen más canciones y mayor probabilidad de éxito |
| **Eliminación de ceros de audio** | Potencial | Puede reducir la representación de música poco bailable o de baja energía; medir pérdida por género |
| **Temporalidad** | Metodológico | Sin fecha de extracción, la popularidad puede estar desactualizada |

### Privacidad
El CSV no contiene datos de usuarios. El riesgo de privacidad individual es **bajo**.
Si se incorporaran historiales de reproducción: minimización, pseudonimización, base legal (GDPR/Ley 19.628 Chile).

### Medidas de mitigación
- Evaluar MAE/RMSE **por género** (equidad algorítmica).
- Usar GroupShuffleSplit (implementado).
- Documentar limitaciones para usuarios finales.
- Mantener supervisión humana en decisiones editoriales.
- No comunicar predicciones como medida de calidad artística.


## 17. Conclusiones de la EP1

1. La calidad se evalúa con nulos explícitos, «?», vacíos y ceros bajo reglas documentadas.
2. Las pérdidas se calculan por **filas únicas**, porque las incidencias pueden coincidir.
3. Los álbumes desconocidos se conservan con una etiqueta que no inventa su tipo de lanzamiento.
4. El compás se imputa con una categoría entera observada solo en entrenamiento.
5. Los nombres de columnas, títulos y ejes están en español desde la carga.
6. La partición por canción evita compartir un mismo ID entre entrenamiento y prueba.
7. El pipeline queda preparado para EP2; su rendimiento predictivo aún no ha sido evaluado.

Antes de la presentación, ejecutar todas las celdas con el CSV original y utilizar
las cifras de `balance_limpieza.csv`, la tabla descriptiva y los gráficos recién
generados. Los resultados numéricos anteriores no representan automáticamente
esta nueva limpieza.

**Cobertura:** fuentes (sección 3), preparación (6 y 15), EDA y calidad (7–14),
sesgos y ética (16). La cobertura no equivale a una calificación de la rúbrica.

Documentación del proyecto: `README_Spotify_ML.md`.
